## Elección del modelo de clasificación y extracción de features🔬 

En este notebook testeo posibles algoritmos de clasificación una vez obtenidas las caras vectorizadas y con sus respectivos labels de emoción.

Pruebo los distintos modelos...

- Supervisado:
    - KNN
    - SVM

In [ ]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from imblearn.over_sampling import RandomOverSampler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

#### Carga de datos

In [15]:
def _load_faces(dir: str) -> pd.DataFrame:    
    angry_faces = pd.read_csv(f"{dir}/angry_faces.csv", header=None).values
    happy_faces = pd.read_csv(f"{dir}/happy_faces.csv", header=None).values
    sad_faces = pd.read_csv(f"{dir}/sad_faces.csv", header=None).values
    surprised_faces = pd.read_csv(f"{dir}/surprise_faces.csv", header=None).values
    neutral_faces = pd.read_csv(f"{dir}/neutral_faces.csv", header=None).values
    disgusted_faces = pd.read_csv(f"{dir}/disgust_faces.csv", header=None).values

    #concat every  face into a single matrix with its corresponding label
    _faces = np.concatenate([
        angry_faces,
        happy_faces,
        sad_faces,
        surprised_faces,    
        neutral_faces,
        disgusted_faces
    ], axis=0)

    #labels for each face
    labels = np.concatenate([
        np.full(angry_faces.shape[0], "angry"),
        np.full(happy_faces.shape[0], "happy"),
        np.full(sad_faces.shape[0], "sad"),
        np.full(surprised_faces.shape[0], "surprised"),
        np.full(neutral_faces.shape[0], "neutral"),
        np.full(disgusted_faces.shape[0], "disgusted")
    ], axis=0)

    faces_df = pd.DataFrame(_faces)

    # if last column is not label, add it
    if faces_df[faces_df.columns[-1]].dtype != object:
        faces_df['label'] = labels
    else:
        faces_df.rename(columns={faces_df.columns[-1]: 'label'}, inplace=True)

    return faces_df

faces = _load_faces("../data/faces")        
pca_faces = _load_faces("../data/pca_faces")
lbp_faces = _load_faces("../data/lbp_faces")

features = {
    'faces': faces,
    'pca_faces': pca_faces,
    'lbp_faces': lbp_faces
}

#### Comparación de features

In [ ]:
features_table = {}
for feature_name, data in features.items():
    num_rows, num_features = data.shape
    features_table[feature_name] = {
        'num_rows': num_rows,
        'num_features': num_features
    }

pd.DataFrame(features_table).T

,num_rows,num_features
faces,38528,2305
pca_faces,9351,1001
lbp_faces,49224,257


#### Comparación de modelos

In [ ]:
def run_model(
    pca_active: bool,
    normalize_X: bool,
    data: pd.DataFrame,
    model
):
    # extract features and labels
    X, Y = data.iloc[:, :-1].values, data['label'].values

    if _pca_active:
        pca = PCA(n_components=100)
        X = pca.fit_transform(X)

    # balance classes
    ros = RandomOverSampler(random_state=30)
    X_bal, Y_bal = ros.fit_resample(X, Y)

    # split into train and test
    X_train, X_test, y_train, y_test = train_test_split(
        X_bal, 
        Y_bal, 
        test_size=0.2, 
        random_state=30,
        stratify=Y_bal
    )

    if _normalize_X:
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)  
    
    # train
    model.fit(X_train, y_train)

    # evaluate
    score = model.score(X_test, y_test)
    return score    

In [21]:
knn = KNeighborsClassifier(n_neighbors=6)
svm = SVC(
    kernel='rbf',
    C=1.0,
    gamma='scale',
    max_iter=1000,   
    random_state=42
)

models = {
    'knn': knn,
    'svm': svm
}

In [ ]:
table_models = []
for feature_name, data in features.items():
    for model_name, model in models.items():
        for _pca_active in [True, False]:
            for normalize_X in [True, False]:
                score = run_model(
                    pca_active=_pca_active,
                    normalize_X=normalize_X,
                    data=data,
                    model=model
                )
                table_models.append({
                    'feature': feature_name,
                    'model': model_name,
                    'score': score,
                    'pca_active': _pca_active,
                    'normalize_X': normalize_X
                })

pd.DataFrame(table_models).sort_values(by='score', ascending=False)